In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LinearRegression

For questions 1 and 2: 

Do a regression to estimate the fixed effect of each group. We assume that there is one single linear coefficient for all the data, plus the fixed effect of each group. Use the file homework_2.1.csv.  The variables G1, G2, and G3 are the outcomes and the time is the treatment.

Question 1


Which of these is closest to being the coefficient of group 1? 

In [10]:
from pathlib import Path
from sklearn.linear_model import LinearRegression

def find_csv(name):
    return next(p / 'data' / name for p in [Path.cwd(), *Path.cwd().parents]
                if (p / 'data' / name).exists())

df1 = pd.read_csv(find_csv('homework_2.1.csv'))
df1.head()

,time,G1,G2,G3
0,0,0.882026,1.441575,0.065409
1,1,0.210079,-0.163880,0.140310
2,2,0.509369,-0.115242,0.819830
3,3,1.150447,1.014698,0.607632
4,4,0.973779,-0.046562,0.610066


In [11]:
# Wide -> long: one row per (time, group)
long_df = df1.melt(id_vars='time', var_name='group', value_name='y')

# Design matrix: shared time slope + one dummy per group
X_fe = pd.concat([long_df[['time']],
                  pd.get_dummies(long_df['group']).astype(float)], axis=1)

fe_model = LinearRegression(fit_intercept=False).fit(X_fe, long_df['y'])
coefs = dict(zip(X_fe.columns, fe_model.coef_))
print(coefs)
print("Group 1 fixed effect:", coefs['G1'])


for g in ['G1', 'G2', 'G3']:
    slope = LinearRegression().fit(df1[['time']], df1[g]).coef_[0]
    print(g, slope)

{'time': np.float64(0.009017213376688269), 'G1': np.float64(0.07855194562167311), 'G2': np.float64(0.5896544232284037), 'G3': np.float64(0.26903180757269896)}
Group 1 fixed effect: 0.07855194562167311
G1 0.00849834916875009
G2 0.007080468023267677
G3 0.011472822938047049




---

### The idea
The data is in **wide** format: one `time` column and one outcome column for each group (`G1`, `G2`, `G3`). The model you want is:


$$y_{it} = \beta \cdot \text{time}_t + \alpha_i + \varepsilon_{it}$$

- $\beta$ is the **single slope shared by all groups** (Question 2).
- $\alpha_i$ is each group's **fixed effect**, its own intercept (Question 1 asks for $\alpha_{G1}$).

To fit it:
1. **Reshape to long format** (`melt`), so each row is one (time, group, y) observation.
2. **One-hot encode the group**, so each group gets its own intercept column.
3. **Fit with `fit_intercept=False`.** The three dummy columns already act as intercepts. Adding a global intercept as well creates perfect collinearity (the dummy variable trap).



---




Question 2


Which of these is closest to being the common linear coefficient for all groups?

In [12]:
print("Common linear coefficient (time):", coefs['time'])

Common linear coefficient (time): 0.009017213376688269


| | Value |
|---|---|
| **Q1: G1 fixed effect** | **≈ 0.079** |
| Q2: common slope on `time` | **≈ 0.009** |
| (for reference) G2, G3 | ≈ 0.590, ≈ 0.269 |

For questions 3-5:

Given a data set, create a bootstrap simulation to try different possibilities. 
Use the file homework_2.2.csv 

Question 3

If we were to measure the effect of the treatment simply by subtracting the outcomes of the treated vs. untreated population, which of these is closest to the mean effect? (This is not the recommended way of measuring the mean effect when there are confounders!) 

The columns are `X` (treatment: 1,972 treated vs 8,028 untreated), `Y` (outcome) and `Z` (confounder).

In [13]:
df2 = pd.read_csv(find_csv('homework_2.2.csv'))

naive_effect = df2[df2['X'] == 1]['Y'].mean() - df2[df2['X'] == 0]['Y'].mean()
print("Naive mean effect:", naive_effect)

Naive mean effect: 2.920717264723189


Question 4


If we were to use bootstrap sampling to measure the variance of that effect, again finding the effect using the non-recommended approach, which of these is closest to that variance?

In [14]:
rng = np.random.default_rng(0)
n = len(df2)
n_boot = 1000

naive_effects = []
for _ in range(n_boot):
    s = df2.iloc[rng.integers(0, n, n)]          # resample rows with replacement
    naive_effects.append(s[s['X'] == 1]['Y'].mean() - s[s['X'] == 0]['Y'].mean())

print("Bootstrap variance:", np.var(naive_effects))

Bootstrap variance: 0.028787267041605325


Bootstrap means: **resample all rows with replacement, at the same size as the data**, recompute the statistic, and repeat many times. The spread of those repeated estimates approximates the sampling variability.

**≈ 0.029.** The exact value shifts a little with the seed, but stays around 0.03.

---

Question 5


if we ran a linear regression (with intercept) to measure the effect, which of these is closest to the skewness of the effect measured? (Look up skewness online. You can use scipy.stats.skew to compute the skewness of a list of numbers.) 

In [15]:
from scipy.stats import skew

reg_effects = []
for _ in range(n_boot):
    s = df2.iloc[rng.integers(0, n, n)]
    reg_effects.append(LinearRegression().fit(s[['X']], s['Y']).coef_[0])

print("Skewness:", skew(reg_effects))

Skewness: 0.09338484076680005


Use the same bootstrap loop. In each resample, fit `Y ~ X` with an intercept and keep the coefficient on `X`. Then compute the skewness of those coefficients.

**The skewness is ≈ 0**; I got about 0.1–0.16, and it varies with the seed. Pick the option closest to 0.

**Why it's ≈ 0:** with 10,000 rows, the central limit theorem makes the bootstrap distribution of the coefficient almost normal, and a normal distribution has zero skew. The leftover noise comes from using only 1,000 bootstrap draws. Skewness estimated from 1,000 draws has a standard error of about \(\sqrt{6/1000} \approx 0.08\), so values like 0.1 are consistent with a true skew of 0.

If you add `Z` to the regression (`s[['X','Z']]`), the skewness is still ≈ 0.11, so the answer is the same either way.

---

### Sanity checks
- With a single binary regressor, the regression coefficient on `X` equals the naive difference in means exactly. So in the Q5 loop, the coefficients should average around 2.92.
- If Q1 gives you a strange value, check that `fit_intercept=False` is set. Leaving the default intercept on changes how the group coefficients come out.